There were ISs that seem to be lost in the next generation.
I here create a batch script to display the reads that contain the lost ISs.

In [1]:
import pandas as pd
import re, io, subprocess
import os, sys

from Bio import SeqIO
from Bio.Seq import Seq
import pysam

In [19]:
file_list_path = '../data/File_list_20250204.csv'
ex_df_path = '../data/excision_status.3000.csv'
is_pos_path = '../data/IS_positions.csv'
ref_fasta_dir = '../tmp/clean_fasta/'
bat_dir = '../exp/IGV_script/ISExcision/'
bed_dir = '../exp/dat/bed/'
win_base_dir = r'V:\2022\analysis_2023\20250112_Revise\for_publication'
igv_snapshot_dir = win_base_dir + r'\exp\fig\IGV\Excision'
igv_fasta_dir = win_base_dir + r'\tmp\clean_fasta'
igv_bam_dir = win_base_dir + r'\exp\bam\mm2_trimmed\all'
igv_bam_dir2 = win_base_dir + r'\exp\bam\mm2_cutadapt\all'
igv_bed_dir = win_base_dir + r'\exp\dat\bed'

os.makedirs(bat_dir, exist_ok=True)
os.makedirs(bed_dir, exist_ok=True)
igv_snapshot_dir_ = '../exp/fig/IGV/Excision'
os.makedirs(igv_snapshot_dir_, exist_ok=True)

tmp_dir = '../tmp/ISExcision'

os.makedirs(tmp_dir, exist_ok=True)

In [3]:
ids = pd.read_csv(file_list_path)
ids.head(2)

,IS_Detect_ID,ParentLine,SubLine,gen,file_name,Anc,RecA,Prefix,sample_name_raw,Contig_Date,Complete,Folder,Folder_check,folder_err,File
0,NaN,0,0,Anc,20230904/MDS42_IS1.fa,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,L01_Anc,1,1,FACS,20231004/L01_Anc_m1.fasta,R01,1,L01-1,L01_Anc,20220418,True,20231004.0,20231004.0,NaN,L01_Anc_m1.fasta


In [4]:
is_pos_df = pd.read_csv(is_pos_path)
is_pos_df.head(2)

,start,end,IS_strand,max_alignment_length,length,cluster_id,Line,Gen
0,476928,480020,reverse,3093,3093,0,L01-1,1
1,557167,560259,reverse,3093,3093,1,L01-1,1


In [7]:
ex_df = pd.read_csv(ex_df_path)
ex_df.head(2)

,Line,Gen,is_cluster_id,min_dist,head_qend,tail_qstart,min_i,min_j,raw_dist,min_start_raw,min_end_raw,flank_dist,dir
0,L01-1,1,0,3093,2999,1,0,1,3094,476913,480007,3093,True
1,L01-1,1,1,8024,2063,311,2,3,9269,547586,555609,-8024,False


In [10]:
ex_df_sub = ex_df.query('min_dist < 100')
ex_df_sub = ex_df_sub.query('not(Line == "L04-1" and Gen == 2)')
print(ex_df_sub.shape)
ex_df_sub.head(2)

(26, 13)


,Line,Gen,is_cluster_id,min_dist,head_qend,tail_qstart,min_i,min_j,raw_dist,min_start_raw,min_end_raw,flank_dist,dir
72,L01-3,2,10,7,2998,1,616,617,-5,1932254,1932248,-7,True
82,L01-4,1,1,7,2998,1,656,658,-5,557155,557149,-7,True


In [6]:
ids.gen.unique()

array(['Anc', 'FACS', '8', '20'], dtype=object)

In [12]:
is_pos_df.head(2)

,start,end,IS_strand,max_alignment_length,length,cluster_id,Line,Gen
0,476928,480020,reverse,3093,3093,0,L01-1,1
1,557167,560259,reverse,3093,3093,1,L01-1,1


In [21]:
min_igv_range_size = 20000

def write_script_sample(bat_file, sample_, vis_range_start, vis_range_end, clip_id):
	# Generate IGV commands
	bat_file.write("new\n")
	bat_file.write(f"genome {igv_fasta_dir}\\{sample_}.fasta\n")
	bat_file.write(f"load {igv_bam_dir}\\{sample_}.bam\n")
	#bat_file.write(f"load {igv_bam_dir2}\\{sample_}.bam\n")
	bat_file.write(f"load {igv_bed_dir}\\{sample_}.bed\n")
	bat_file.write(f"goto {sample_}:{vis_range_start}-{vis_range_end}\n")
	bat_file.write(f"collapse {sample_}.bam\n")
	bat_file.write(f"maxPanelHeight 600\n")
	bat_file.write(f"expand {sample_}.bam\n")
	bat_file.write(f"setColor 155,155,155 {sample_}.bam\n")
	#bat_file.write(f"setTrackHeight 1000 {sample_}.bam\n")
	bat_file.write(f"setTrackHeight 15 {sample_}.bed\n")
	bat_file.write(f"snapshot {clip_id}.png\n")
	bat_file.write(f"snapshot {clip_id}.svg\n")

	# fullpath
	bat_file.write(f"savesession {igv_snapshot_dir}\\{clip_id}.xml\n")
	bat_file.write(f"\n\n")

bat_file_dir = os.path.join(bat_dir, 'ISExcisionRelatedReads.bat')
with open(bat_file_dir, "w") as bat_file:
	# Write the IGV setup commands
	bat_file.write(f'snapshotDirectory {igv_snapshot_dir}\n')
	bat_file.write("preference DEFAULT_FONT_SIZE 20\n")
	for row in ex_df_sub.itertuples():
		# the existing IS
		run_id = f'{row.Line}.{row.Gen}.{row.is_cluster_id}_01'
		line_, gen_ = row.Line, row.Gen
		sample_ = f'{row.Line}_' + ('G08' if gen_ == 2 else 'G20' if gen_ == 3 else 'Anc')
		sample_ = sample_ if gen_ > 1 else row.Line.split('-')[0] + '_Anc'
		print(sample_)
  
		is_pos_row = is_pos_df.query(f'cluster_id == {row.is_cluster_id} and Line == "{row.Line}" and Gen == {row.Gen}')
		vis_range_mid = (is_pos_row.start.min() + is_pos_row.end.max()) // 2
		vis_range_start = vis_range_mid - min_igv_range_size//2
		vis_range_end = vis_range_mid + min_igv_range_size//2

		bed_df = is_pos_df.query(f'Line == "{line_}" & Gen == {gen_}')
		bed_df['chr'] = sample_
		bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
		bed_df['IS_ID'] = bed_df.cluster_id.apply(lambda x: f'{line_}.{gen_}.{x}')
		bed_df['start'] = bed_df['start'] - 1
		bed_df = bed_df[['chr', 'start', 'end', 'IS_ID', 'length', 'strand']]
		bed_df.to_csv(os.path.join(bed_dir, f'{sample_}.bed'), sep='\t', header=False, index=False)

		write_script_sample(bat_file, sample_, vis_range_start, vis_range_end, run_id)

		# position in next gen
		run_id = f'{row.Line}.{row.Gen}.{row.is_cluster_id}_02'
		line_, gen_ = row.Line, row.Gen + 1
		sample_ = f'{row.Line}_' + ('G08' if gen_ == 2 else 'G20' if gen_ == 3 else 'Anc')
		sample_ = sample_ if gen_ > 1 else row.Line.split('-')[0] + '_Anc'
		print(sample_)
  
		vis_range_mid = (row.min_start_raw + row.min_end_raw) // 2
		vis_range_start = vis_range_mid - min_igv_range_size//2
		vis_range_end = vis_range_mid + min_igv_range_size//2

		# create bed file for IGV with strand
		bed_df = is_pos_df.query(f'Line == "{line_}" & Gen == {gen_}')
		bed_df['chr'] = sample_
		bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
		bed_df['IS_ID'] = bed_df.cluster_id.apply(lambda x: f'{line_}.{gen_}.{x}')
		bed_df['start'] = bed_df['start'] - 1
		bed_df = bed_df[['chr', 'start', 'end', 'IS_ID', 'length', 'strand']]
		bed_df.to_csv(os.path.join(bed_dir, f'{sample_}.bed'), sep='\t', header=False, index=False)

		write_script_sample(bat_file, sample_, vis_range_start, vis_range_end, run_id)

L01-3_G08
L01-3_G20
L01_Anc
L01-4_G08
L02-1_G08
L02-1_G20
L02-3_G08
L02-3_G20
L02-4_G08
L02-4_G20
L02-4_G08
L02-4_G20
L03-3_G08
L03-3_G20


/tmp/ipykernel_20893/1002094256.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['chr'] = sample_
/tmp/ipykernel_20893/1002094256.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
/tmp/ipykernel_20893/1002094256.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

L03-3_G08
L03-3_G20
L03-4_G08
L03-4_G20
L03-4_G08
L03-4_G20
L04_Anc
L04-1_G08
L04-3_G08
L04-3_G20
L04-4_G08
L04-4_G20
L05-1_G08
L05-1_G20


/tmp/ipykernel_20893/1002094256.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['chr'] = sample_
/tmp/ipykernel_20893/1002094256.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
/tmp/ipykernel_20893/1002094256.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

L05_Anc
L05-2_G08
L05-4_G08
L05-4_G20
L06-3_G08
L06-3_G20
L07-4_G08
L07-4_G20
L08-4_G08
L08-4_G20
L09_Anc
L09-2_G08
L09-2_G08
L09-2_G20


/tmp/ipykernel_20893/1002094256.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['chr'] = sample_
/tmp/ipykernel_20893/1002094256.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
/tmp/ipykernel_20893/1002094256.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

L09-3_G08
L09-3_G20
L09-4_G08
L09-4_G20
L10-1_G08
L10-1_G20
L10-3_G08
L10-3_G20
L11_Anc
L11-2_G08


/tmp/ipykernel_20893/1002094256.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['chr'] = sample_
/tmp/ipykernel_20893/1002094256.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
/tmp/ipykernel_20893/1002094256.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://